In [ ]:
# ===== 패키지 설치 (최초 1회만 실행) =====
!pip install python-dotenv
!pip install -U langchain langchain-openai langchain-teddynote

# ===== 필요한 모듈 import =====
import os
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, load_prompt
from datetime import datetime

# ===== 환경변수(.env) 로드 =====
load_dotenv()  # override=False가 기본값이므로 시스템 환경변수가 우선순위를 가짐

# (선택) 정상 로드 확인
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

# ===== LangSmith 추적 시작 =====
logging.langsmith("CH02-Prompt")  # 프로젝트명 입력

# ===== LLM 객체 생성 =====
llm = ChatOpenAI()

### YAML 파일로부터 프롬프트 템플릿 로드하기

In [13]:
_type: "prompt"
template: "{fruit}의 색깔이 뭐야?"
input_variables: ["fruit"]

In [14]:
from langchain_core.prompts import load_prompt
prompt = load_prompt("prompts/fruit_color.yaml", encoding="utf-8")
prompt

PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색깔이 뭐야?')

In [15]:
prompt.format(fruit="사과")

'사과의 색깔이 뭐야?'

아래는 `prompts/capital.yaml` 파일의 내용입니다 (YAML 문법이라 파이썬 코드 셀에서 직접 실행하면 안 됩니다):

```yaml
_type: "prompt"
template: |
    {country}의 수도에 대해서 알려주세요.
    수도의 특징을 다음의 양식에 맞게 정리해 주세요.
    300자 내외로 작성해 주세요.
    한글로 작성해 주세요.
    ----
    [양식]
    1. 면적
    2. 인구
    3. 역사적 장소
    4. 특산품
        #Answer:
input_variables: ["country"]
```

In [17]:
prompt2 = load_prompt("prompts/capital.yaml", encoding="utf-8")
print(prompt2.format(country="대한민국"))

대한민국의 수도에 대해서 알려주세요.
수도의 특징을 다음의 양식에 맞게 정리해 주세요.
300자 내외로 작성해 주세요.
한글로 작성해 주세요.
----
[양식]
1. 면적
2. 인구
3. 역사적 장소
4. 특산품
    #Answer:



In [18]:
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote.messages import stream_response

chain = prompt2 | ChatOpenAI(model_name="gpt-4o-mini", temperature=0) | StrOutputParser()

answer = chain.stream({"country":"대한민국"})
stream_response(answer)

1. 면적: 서울특별시는 약 605.21㎢의 면적을 가지고 있습니다.  
2. 인구: 2023년 기준으로 서울의 인구는 약 9백만 명에 달합니다.  
3. 역사적 장소: 경복궁, 창덕궁, 남산, 종묘 등 다양한 역사적 장소가 있어 한국의 전통과 문화를 체험할 수 있습니다.  
4. 특산품: 서울의 특산품으로는 한방차, 전통주, 그리고 다양한 길거리 음식들이 유명합니다. 특히, 떡볶이와 순대는 서울을 대표하는 먹거리로 많은 사랑을 받고 있습니다.